# Liber · Block 1 검출기 본학습 (Colab)

Liber의 책 검출기(**YOLO11n-OBB**)를 Colab GPU에서 **데이터 수집 → 1~3권 사진 보강 → 본학습 → 권수 구간별 평가**까지 한 번에 돌리고,
결과 가중치를 Google Drive에 저장한다. 맥북 노트북(`liber_deprehensus.ipynb`)은 이 가중치를 받아 Block 1부터 그대로 쓴다.

### 왜 다시 학습하나
기존 학습 데이터 7,457장 중 **1~3권 사진이 271장(3.6%)**, 6권 이상이 92%였다 → 책이 1~3권뿐인 사진을 잘 못 찾음.

| 보강 | 내용 |
|---|---|
| ① 이전 프로젝트 합성 데이터 | 책등·표지 합성 사진 (1권 사진이 많음). 라벨 2종 → `book` 1종 |
| ② Open Images `Book` | **1권 이상** 사진을 권수 구간(1·2·3·4+)별 할당량대로 |
| ③ 1~3권 잘라내기 | 빽빽한 책장 사진에서 이웃한 1~3권만 잘라 새 사진 + 라벨 (학습용 사진에서만) |
| ④ 1~3권 사진 반복 | 학습 중 더 자주 보이게 복사본 추가 |
| ⑤ 입력 960 · 회전 ±30° | 웹앱 검출기 입력(960)과 맞춤 · 책상에 놓인 표지 대응 |

### 실행 전 준비 (처음 한 번)
1. **런타임 → 런타임 유형 변경 → A100 GPU** (없으면 L4·T4도 동작, 더 느림)
2. 왼쪽 🔑 **보안 비밀(Secrets)** → 이름 `ROBOFLOW_API_KEY`, 값 = Roboflow Private API Key → **노트북 액세스 켜기**
3. (권장) 맥에서 이전 프로젝트 합성 데이터를 압축해 Drive `내 드라이브/liber/prev_books_obb.zip`에 올림
   → 없으면 GitHub에 올라가 있는 예전 버전(1,600장)을 받아 씀
4. **런타임 → 모두 실행**

### 결과 (Drive `내 드라이브/liber/`)
- `det_obb_best.pt` ← **맥으로 가져갈 가중치**
- `runs/det_obb/` 학습 곡선·검증 그림 · `eval_by_count.csv` 권수 구간별 평가
- `books_obb_built.zip` 만든 데이터셋 (세션이 끊겨 다시 시작할 때 재사용)

예상 시간 (A100): 데이터 준비 20~40분 + 학습 에폭당 2~3분 × 최대 150에폭(20에폭 동안 좋아지지 않으면 조기 종료) → 보통 3~6시간

In [ ]:
# Cell 1 · 환경 준비: GPU · 패키지 · Google Drive · 경로
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.183 roboflow==1.7.1

import json
import os
import random
import re
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from google.colab import drive, userdata

drive.mount("/content/drive")

SEED = 42
TIMES = {}   # 단계별 걸린 시간(분) → 마지막 셀에서 표로
random.seed(SEED)
np.random.seed(SEED)

DRIVE = Path("/content/drive/MyDrive/liber")   # 입력·결과 보관 (세션이 끊겨도 남음)
WORK = Path("/content/work")                    # 데이터 작업 공간 (Colab 로컬 디스크: 빠르지만 세션이 끝나면 지워짐)
RAW, SRC, DS = WORK / "raw", WORK / "src", WORK / "books_obb"
for d in (DRIVE, RAW, SRC):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
free_gb = shutil.disk_usage("/content/drive").free / 1e9
print(f"Drive 여유 공간 약 {free_gb:.1f}GB (이 노트북이 Drive에 쓰는 양: 약 0.1GB, SAVE_BUILT=True면 +수 GB)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 A100 GPU로 바꾸세요")

In [ ]:
# Cell 2 · 설정 (여기만 바꾸면 됨)
QUICK_CHECK = False   # True: 데이터 5% · 1에폭으로 셀이 끝까지 도는지만 확인 (약 10분)
REBUILD_DATA = False  # True: Drive에 저장된 데이터셋이 있어도 처음부터 다시 만듦

TRAIN = dict(
    epochs=150,        # 최대 에폭 (patience 동안 좋아지지 않으면 그 전에 멈춤)
    patience=20,
    imgsz=960,         # 웹앱 검출기 입력과 같게
    batch=32,          # A100 40GB 기준. 메모리 부족이면 16
    workers=8,
    degrees=30.0,      # 회전 ±30° (책상에 놓인 표지 · 기울여 찍은 책장)
    fliplr=0.5,
    flipud=0.0,        # 위아래 뒤집힌 책장은 없음
    mosaic=1.0,
    close_mosaic=15,   # 마지막 15에폭은 모자이크 없이 실제 사진 그대로
    save_period=-1,    # 중간 에폭 가중치(epoch10.pt …)는 저장 안 함 → Drive 용량 절약. last.pt·best.pt는 매 에폭 덮어씀
    cache="disk",      # 사진을 학습 크기로 줄여 로컬 디스크에 저장해 두고 읽음 (JPEG 해독 병목 해소)
)                      # "ram"은 쓰지 않음: 사진 읽는 작업 프로세스(workers)마다 캐시가 복사돼 RAM 83GB를 넘겨 세션이 다운됨
import psutil
RAM_GB = psutil.virtual_memory().total / 1e9
SAVE_BUILT = True      # 만든 데이터셋(약 3~5GB)을 Drive에 저장 → 세션이 끊겼을 때 데이터 준비 20~40분을 아낌. Drive 여유 공간이 5GB 미만이면 False
OI_QUOTA = {1: 1500, 2: 1000, 3: 800, 4: 1200}   # Open Images: 사진 속 책 수 구간(4 = 4권 이상)별 최대 장수
FEW_PER_IMAGE = 1      # ③ 빽빽한 책장 사진 1장에서 만들 1~3권 사진 수
FEW_MAX = 3000         # ③ 최대 장수
FEW_REPEAT = 2         # ④ 학습용 1~3권 사진을 몇 배로 볼지 (2 = 복사본 1개 추가)
VAL_RATIO = 0.1
MAX_PER_GROUP = 3      # Roboflow 증강 사본: 원본 1장당 학습용 최대 장수

ROBOFLOW_SOURCES = [   # (workspace, project, version) — version=None이면 최신
    ("book-detection", "book-spine-detection-p2", None),
    ("minhaz", "book_spine_instance_segmentation-uqybb", None),
    ("emptybookslotdetect", "book-spine-seg", None),
    ("libvision", "book-spines-vh9b8", None),
]
PREV_ZIP = DRIVE / "prev_books_obb.zip"
BUILT_ZIP = DRIVE / "books_obb_built.zip"
RUN_DIR = DRIVE / "runs" / ("det_obb_quick" if QUICK_CHECK else "det_obb")
BEST_OUT = DRIVE / ("det_obb_quick.pt" if QUICK_CHECK else "det_obb_best.pt")   # 빠른 확인 결과가 본학습 가중치를 덮어쓰지 않게

if QUICK_CHECK:
    TRAIN.update(epochs=1, patience=1)
    OI_QUOTA = {k: v // 20 for k, v in OI_QUOTA.items()}
    FEW_MAX = 150

# 저장해 둔 데이터셋이 있으면 복원하고 데이터 셀(3~7)을 건너뜀
DATA_READY = False
if BUILT_ZIP.exists() and not REBUILD_DATA and not QUICK_CHECK:
    print(f"저장된 데이터셋 복원: {BUILT_ZIP}")
    subprocess.run(["unzip", "-q", "-o", str(BUILT_ZIP), "-d", str(WORK)], check=True)
    cfg = yaml.safe_load(open(DS / "data.yaml"))
    cfg["path"] = str(DS)
    yaml.safe_dump(cfg, open(DS / "data.yaml", "w"), sort_keys=False)
    DATA_READY = True
print(f"RAM {RAM_GB:.0f}GB → 학습 데이터 캐시: {TRAIN['cache']}")
print("데이터 셀 건너뜀" if DATA_READY else "데이터를 새로 만듭니다 (Cell 3~7)")

In [ ]:
# Cell 3 · 공통 함수: 라벨 변환 · 읽기/쓰기 · 중복 판정 · 원본 묶음
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def to_obb(values, w, h):
    """YOLO 라벨 한 줄(클래스 제외) → 정규화 4꼭짓점. 수평 상자(4값)·다각형·OBB(8값) 모두"""
    if len(values) == 4:
        cx, cy, bw, bh = values
        pts = np.array([[cx - bw / 2, cy - bh / 2], [cx + bw / 2, cy - bh / 2],
                        [cx + bw / 2, cy + bh / 2], [cx - bw / 2, cy + bh / 2]])
    elif len(values) >= 6 and len(values) % 2 == 0:
        poly = (np.array(values).reshape(-1, 2) * [w, h]).astype(np.float32)
        pts = cv2.boxPoints(cv2.minAreaRect(poly)) / [w, h]
    else:
        return None
    pts = np.clip(pts, 0, 1)
    if cv2.contourArea(pts.astype(np.float32)) < 1e-5:
        return None
    return pts.reshape(-1).tolist()


def write_obb_label(path, obbs):
    path.write_text("\n".join("0 " + " ".join(f"{v:.6f}" for v in o) for o in obbs) + "\n")


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in lbl_path.read_text().splitlines() if len(l.split()) >= 9]


def dhash(path, size=8):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits(g[:, 1:] > g[:, :-1]).tobytes()


def group_key(src_name, img):
    """같은 원본에서 나온 사진끼리 묶는 키 (Roboflow 증강 사본 '<분할>_<원본>_jpg.rf.<해시>' → '<원본>_jpg')"""
    if src_name.startswith("rf_"):
        return re.sub(r"^(train|valid|test)_", "", img.stem).split(".rf.")[0]
    return img.stem


def save_pair(img_src, obbs, out_dir, stem):
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_src, out_dir / "images" / f"{stem}{Path(img_src).suffix.lower()}")
    write_obb_label(out_dir / "labels" / f"{stem}.txt", obbs)

In [ ]:
# Cell 4 · [데이터 ①] Roboflow 책등 데이터 (CC BY 4.0) → OBB 라벨
from roboflow import Roboflow

t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    summary = []
    for ws, proj, ver in ROBOFLOW_SOURCES:
        name = f"rf_{proj}"
        try:
            project = rf.workspace(ws).project(proj)
            if ver is None:
                ver = max(int(str(v.version).split("/")[-1]) for v in project.versions())
            project.version(ver).download("yolov8", location=str(RAW / name), overwrite=False)
        except Exception as e:
            print(f"[skip] {ws}/{proj}: {e}")
            continue
        n_img = n_box = 0
        for k, img_path in enumerate(sorted((RAW / name).rglob("*"))):
            if img_path.suffix.lower() not in IMG_EXTS or img_path.parent.name != "images":
                continue
            lbl = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
            if not lbl.exists():
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            obbs = [o for l in lbl.read_text().splitlines() if len(l.split()) >= 5
                    for o in [to_obb([float(v) for v in l.split()[1:]], w, h)] if o]
            if obbs:
                # 원본 파일명이 255바이트를 넘기도 해서 번호로 저장하고, 묶음 키는 원래 이름으로 기록
                stem = f"{k:06d}__{group_key(name, img_path)[:80]}"
                save_pair(img_path, obbs, SRC / name, stem)
                n_img += 1
                n_box += len(obbs)
        summary.append({"source": name, "version": ver, "images": n_img, "boxes": n_box})
    display(pd.DataFrame(summary))
TIMES["① Roboflow 다운로드·변환"] = (time.time() - t0) / 60

In [ ]:
# Cell 5 · [데이터 ②] Open Images V7 'Book' — 1권 이상, 권수 구간별 할당량
#   - 책 묶음(IsGroupOf)·그림 속 책(IsDepiction) 상자가 있는 사진은 제외 (묶음 안의 책은 라벨이 없어 '책 아님'으로 학습됨)
#   - 라벨은 수평 상자 → 회전각 0 사각형
OI_BASE = "https://storage.googleapis.com/openimages"
OI_CSVS = {"train": f"{OI_BASE}/v6/oidv6-train-annotations-bbox.csv",
           "validation": f"{OI_BASE}/v5/validation-annotations-bbox.csv",
           "test": f"{OI_BASE}/v5/test-annotations-bbox.csv"}
BOOK_MID = "/m/0bt_c3"
COLS = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax", "IsGroupOf", "IsDepiction"]

t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    csv_dir = RAW / "oi_csv"
    csv_dir.mkdir(exist_ok=True)
    frames = []
    for split, url in OI_CSVS.items():
        f = csv_dir / Path(url).name
        if not f.exists():
            subprocess.run(["wget", "-q", "-O", str(f), url], check=True)   # train 주석은 약 2.2GB
        for chunk in pd.read_csv(f, usecols=COLS, chunksize=2_000_000):
            chunk = chunk[chunk.LabelName == BOOK_MID]
            if len(chunk):
                frames.append(chunk.assign(split=split))
    books = pd.concat(frames, ignore_index=True)
    per = books.groupby(["split", "ImageID"]).agg(n=("ImageID", "size"), group=("IsGroupOf", "max"),
                                                  dep=("IsDepiction", "max")).reset_index()
    per = per[(per.group != 1) & (per.dep != 1)]
    per["bucket"] = per.n.clip(upper=4)
    picks = pd.concat([g.sample(n=min(OI_QUOTA[b], len(g)), random_state=SEED) for b, g in per.groupby("bucket")])
    print("구간별 후보 / 선택:", {int(b): (len(g), int((picks.bucket == b).sum())) for b, g in per.groupby("bucket")})
    boxes_by_id = books[books.ImageID.isin(picks.ImageID)].groupby("ImageID")

    out = SRC / "openimages"
    (out / "images").mkdir(parents=True, exist_ok=True)
    (out / "labels").mkdir(parents=True, exist_ok=True)

    def fetch(row):
        dst = out / "images" / f"{row.ImageID}.jpg"
        if not dst.exists():
            try:
                r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{row.split}/{row.ImageID}.jpg", timeout=30)
                r.raise_for_status()
            except requests.RequestException:
                return False
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                return False
            s = 1280 / max(img.shape[:2])
            if s < 1:
                img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        b = boxes_by_id.get_group(row.ImageID)
        write_obb_label(out / "labels" / f"{row.ImageID}.txt",
                        [[x0, y0, x1, y0, x1, y1, x0, y1] for x0, x1, y0, y1 in b[["XMin", "XMax", "YMin", "YMax"]].itertuples(index=False)])
        return True

    with ThreadPoolExecutor(32) as ex:
        ok = list(ex.map(fetch, picks.itertuples(index=False)))
    print(f"Open Images 저장 {sum(ok):,}/{len(picks):,}장")
TIMES["② Open Images 주석·사진"] = (time.time() - t0) / 60

In [ ]:
# Cell 6 · [데이터 ③ 준비] 이전 프로젝트 합성·실사진 데이터 (책등·표지 2종 → book 1종)
#   - Drive의 prev_books_obb.zip이 있으면 그것(최신), 없으면 GitHub 저장소의 예전 버전
t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    prev = RAW / "prev"
    if not prev.exists():
        if PREV_ZIP.exists():
            subprocess.run(["unzip", "-q", "-o", str(PREV_ZIP), "-d", str(prev)], check=True)
        else:
            print("Drive에 prev_books_obb.zip이 없어 GitHub 예전 버전(합성 1,600장)을 받습니다")
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                            "https://github.com/skytinstone/shelf-ocr-ondevice.git", str(RAW / "repo")], check=True)
            subprocess.run(["git", "-C", str(RAW / "repo"), "sparse-checkout", "set", "datasets/books_obb"], check=True)
            shutil.move(str(RAW / "repo" / "datasets" / "books_obb"), str(prev))
    root = next(p.parent.parent for p in prev.rglob("images/train"))
    n = 0
    for split in ("train", "val"):
        for img_path in sorted((root / "images" / split).glob("*")):
            if img_path.suffix.lower() not in IMG_EXTS or img_path.name.startswith("oi_"):   # Open Images는 Cell 5에서 따로 받음
                continue
            lbl = root / "labels" / split / f"{img_path.stem}.txt"
            if not lbl.exists():
                continue
            obbs = [[float(v) for v in l.split()[1:9]] for l in lbl.read_text().splitlines() if len(l.split()) >= 9]
            if obbs:
                save_pair(img_path, obbs, SRC / "prev", f"{split}_{img_path.stem}")
                n += 1
    print(f"이전 프로젝트 데이터 {n}장 → {SRC / 'prev'}")
TIMES["이전 프로젝트 데이터"] = (time.time() - t0) / 60

In [ ]:
# Cell 7 · [통합] 중복 제거 · 원본 묶음 단위 train/val 분할 · ③ 1~3권 잘라내기 · ④ 1~3권 반복 · data.yaml
#   ③은 학습용(train)으로 나뉜 Roboflow 책장 사진에서만 만든다 → 검증 사진의 일부가 학습에 섞이지 않음


def few_crops(img, polys, rng, n_crops):
    """빽빽한 책장 사진 → 이웃한 1~3권만 담은 사진들. 잘린 책(10~60%만 보임)이 생기는 자리는 버리고 다시 고른다"""
    h, w = img.shape[:2]
    order = sorted(range(len(polys)), key=lambda i: polys[i][:, 0].mean())
    out = []
    for _ in range(n_crops * 10):
        if len(out) >= n_crops:
            break
        k = rng.choice([1, 1, 2, 2, 3])
        if len(order) < k + 1:
            continue
        s = rng.randrange(0, len(order) - k + 1)
        pts = np.concatenate([polys[i] for i in order[s:s + k]])
        (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
        mx, my = rng.uniform(0.02, 0.15) * (x1 - x0), rng.uniform(0.05, 0.3) * (y1 - y0)
        X0, Y0, X1, Y1 = max(0, int(x0 - mx)), max(0, int(y0 - my)), min(w, int(x1 + mx)), min(h, int(y1 + my))
        if X1 - X0 < 32 or Y1 - Y0 < 32:
            continue
        rect = np.float32([[X0, Y0], [X1, Y0], [X1, Y1], [X0, Y1]])
        keep, bad = [], False
        for p in polys:
            area = cv2.contourArea(p)
            if area <= 0:
                continue
            inter, ip = cv2.intersectConvexConvex(p, rect)
            vis = inter / area
            if vis >= 0.6 and ip is not None:
                keep.append(cv2.boxPoints(cv2.minAreaRect(ip.reshape(-1, 2))) - np.float32([X0, Y0]))
            elif vis > 0.1:
                bad = True
                break
        if not bad and 1 <= len(keep) <= 3:
            out.append((img[Y0:Y1, X0:X1].copy(), keep))
    return out


t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    if DS.exists():
        shutil.rmtree(DS)
    for split in ("train", "val"):
        (DS / "images" / split).mkdir(parents=True)
        (DS / "labels" / split).mkdir(parents=True)

    rng = random.Random(SEED)
    seen, rows, n_dup, n_cap = set(), [], 0, 0

    def put(img_path, lbl_path, split, source, group):
        name, ext = f"{source}__{len(rows):06d}", Path(img_path).suffix
        try:
            os.link(img_path, DS / "images" / split / f"{name}{ext}")
        except OSError:
            shutil.copy2(img_path, DS / "images" / split / f"{name}{ext}")
        shutil.copy2(lbl_path, DS / "labels" / split / f"{name}.txt")
        n = len(Path(lbl_path).read_text().splitlines())
        rows.append({"name": name, "ext": ext, "source": source, "split": split, "group": group, "boxes": n})
        return name

    for src_dir in sorted(p for p in SRC.iterdir() if p.is_dir()):
        groups = defaultdict(list)
        for img in sorted((src_dir / "images").iterdir()):
            lbl = src_dir / "labels" / f"{img.stem}.txt"
            h = dhash(img) if lbl.exists() else None
            if h is None:
                continue
            if h in seen:
                n_dup += 1
                continue
            seen.add(h)
            key = img.stem.split("__", 1)[1] if src_dir.name.startswith("rf_") else img.stem
            groups[key].append((img, lbl))
        keys = sorted(groups)
        rng.shuffle(keys)
        if QUICK_CHECK:
            keys = keys[:max(20, len(keys) // 20)]
        n_val = max(1, int(len(keys) * VAL_RATIO)) if keys else 0
        for gi, key in enumerate(keys):
            split = "val" if gi < n_val else "train"
            members = groups[key]
            rng.shuffle(members)
            keep = members[:1 if split == "val" else MAX_PER_GROUP]
            n_cap += len(members) - len(keep)
            for img, lbl in keep:
                put(img, lbl, split, src_dir.name, key)

    # ③ 1~3권 잘라내기 (학습용 Roboflow 책장 사진에서만)
    few_dir = WORK / "fewcrop"
    few_dir.mkdir(exist_ok=True)
    dense = [r for r in rows if r["split"] == "train" and r["source"].startswith("rf_") and r["boxes"] >= 4]
    rng.shuffle(dense)
    n_few = 0
    for r in dense:
        if n_few >= FEW_MAX:
            break
        img_path = DS / "images" / "train" / f"{r['name']}{r['ext']}"
        img = cv2.imread(str(img_path))
        polys = read_obb(DS / "labels" / "train" / f"{r['name']}.txt", img.shape[1], img.shape[0])
        for j, (crop, boxes) in enumerate(few_crops(img, polys, rng, FEW_PER_IMAGE)):
            ch, cw = crop.shape[:2]
            stem = f"{r['name']}_{j}"
            cv2.imwrite(str(few_dir / f"{stem}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            write_obb_label(few_dir / f"{stem}.txt", [(b / np.float32([cw, ch])).clip(0, 1).reshape(-1).tolist() for b in boxes])
            put(few_dir / f"{stem}.jpg", few_dir / f"{stem}.txt", "train", "fewcrop", r["group"])
            n_few += 1

    # ④ 1~3권 학습 사진 반복
    n_rep = 0
    for r in [r for r in rows if r["split"] == "train" and 1 <= r["boxes"] <= 3]:
        for k in range(1, FEW_REPEAT):
            img_path = DS / "images" / "train" / f"{r['name']}{r['ext']}"
            dst = DS / "images" / "train" / f"{r['name']}_rep{k}{r['ext']}"
            try:
                os.link(img_path, dst)
            except OSError:
                shutil.copy2(img_path, dst)
            shutil.copy2(DS / "labels" / "train" / f"{r['name']}.txt", DS / "labels" / "train" / f"{r['name']}_rep{k}.txt")
            n_rep += 1

    yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val", "names": {0: "book"}},
                   open(DS / "data.yaml", "w"), sort_keys=False)
    manifest = pd.DataFrame(rows)
    manifest.to_csv(DS / "manifest.csv", index=False)
    print(f"중복 제거 {n_dup:,} · 사본 제한 제외 {n_cap:,} · ③ 1~3권 잘라내기 {n_few:,} · ④ 반복 복사본 {n_rep:,}")

    TIMES["통합·③④ 보강"] = (time.time() - t0) / 60
    if SAVE_BUILT and not QUICK_CHECK:   # 세션이 끊겨도 다시 만들지 않게 Drive에 저장 (압축 없이 묶기만: 사진은 이미 JPEG)
        t1 = time.time()
        subprocess.run(["zip", "-q", "-r", "-0", str(WORK / "books_obb_built.zip"), "books_obb"], cwd=WORK, check=True)
        try:
            shutil.copy2(WORK / "books_obb_built.zip", BUILT_ZIP)   # 로컬에서 묶은 뒤 한 번에 복사 (Drive에 직접 쓰는 것보다 빠름)
            print(f"데이터셋 저장 → {BUILT_ZIP}")
        except OSError as e:   # Drive 용량 부족이어도 학습은 계속
            BUILT_ZIP.unlink(missing_ok=True)
            print(f"⚠️ 데이터셋을 Drive에 저장하지 못했습니다(용량 부족?) — 학습은 계속합니다: {e}")
        TIMES["데이터셋 Drive 저장"] = (time.time() - t1) / 60

In [ ]:
# Cell 8 · ✅ 데이터 확인: 출처별 장수 · 사진 속 책 수 분포(보강 전 → 후) · 예시
def count_boxes(split):
    out = Counter()
    for f in (DS / "labels" / split).glob("*.txt"):
        out[min(sum(1 for l in open(f) if l.strip()), 6)] += 1
    return out

manifest = pd.read_csv(DS / "manifest.csv")
display(manifest.pivot_table(index="source", columns="split", values="boxes", aggfunc="count", fill_value=0))
tr, va = count_boxes("train"), count_boxes("val")
dist = pd.DataFrame({"train": tr, "val": va}).sort_index()
dist.index = [f"{i}권" if i < 6 else "6권+" for i in dist.index]
dist["train 비율"] = (dist["train"] / dist["train"].sum() * 100).round(1).astype(str) + "%"
print("사진 속 책 수 분포 (보강 전 Liber 학습 데이터: 1~3권 271장 = 3.6%)")
display(dist)

few = [p for p in (DS / "images" / "train").glob("*") if "_rep" not in p.stem]
random.Random(1).shuffle(few)
picked = [p for p in few if p.stem.startswith("fewcrop")][:4] + [p for p in few if not p.stem.startswith("fewcrop")][:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, picked):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for poly in read_obb(DS / "labels" / "train" / f"{p.stem}.txt", w, h):
        cv2.polylines(img, [poly.astype(np.int32)], True, (255, 0, 0), max(2, w // 300))
    ax.imshow(img)
    ax.set_title(p.stem.split("__")[0], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 · [본학습] YOLO11n-OBB — 결과는 Drive에 바로 저장 (세션이 끊기면 이 셀만 다시 실행하면 이어서 학습)
from ultralytics import YOLO

last = RUN_DIR / "weights" / "last.pt"
t0 = time.time()
if last.exists() and not QUICK_CHECK:
    print(f"이어서 학습: {last}")
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:   # 이미 끝까지 학습한 경우
        print(f"이어서 할 학습 없음: {e}")
else:
    model = YOLO("yolo11n-obb.pt")
    model.train(data=str(DS / "data.yaml"), project=str(RUN_DIR.parent), name=RUN_DIR.name, exist_ok=True,
                device=DEVICE, seed=SEED, fraction=0.05 if QUICK_CHECK else 1.0, **TRAIN)
TIMES["본학습"] = (time.time() - t0) / 60
print(f"학습 시간 {(time.time() - t0) / 3600:.2f}시간")

shutil.copy2(RUN_DIR / "weights" / "best.pt", BEST_OUT)
print(f"맥으로 가져갈 가중치 → {BEST_OUT}")

import gc   # 학습용 사진 캐시(RAM 최대 수십 GB)를 비워 다음 단계(ViT 학습 등)가 메모리 부족을 겪지 않게
model = None
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 10 · ✅ 권수 구간별 평가 (검증 사진): 1권·2권·3권 사진에서 실제로 좋아졌는지
#   - 맞춤 기준: 회전 상자 IoU ≥ 0.5 · 검출 신뢰도 ≥ CONF
#   - 권수 정확: 사진 속 책 수를 정확히 맞힌 사진 비율 / 권수 오차: |검출 수 − 실제 수| 평균
CONF = 0.25
best_model = YOLO(str(BEST_OUT))


def poly_iou(a, b):
    inter, _ = cv2.intersectConvexConvex(a, b)
    union = cv2.contourArea(a) + cv2.contourArea(b) - inter
    return inter / union if union > 0 else 0.0


def bucket(n):
    return f"{n}권" if n <= 3 else ("4~9권" if n <= 9 else "10권+")


t0 = time.time()
rows = []
for img_path in sorted((DS / "images" / "val").glob("*")):
    img = cv2.imread(str(img_path))
    gt = read_obb(DS / "labels" / "val" / f"{img_path.stem}.txt", img.shape[1], img.shape[0])
    r = best_model.predict(img, imgsz=TRAIN["imgsz"], conf=CONF, device=DEVICE, verbose=False)[0]
    pred = [p.astype(np.float32) for p in r.obb.xyxyxyxy.cpu().numpy()] if r.obb is not None else []
    used, tp = set(), 0
    for g in gt:
        cand = [(poly_iou(g, p), j) for j, p in enumerate(pred) if j not in used]
        if cand:
            iou, j = max(cand)
            if iou >= 0.5:
                used.add(j)
                tp += 1
    rows.append({"구간": bucket(len(gt)), "실제": len(gt), "검출": len(pred), "맞음": tp})

ev = pd.DataFrame(rows)
order = ["1권", "2권", "3권", "4~9권", "10권+"]
summary = ev.groupby("구간").apply(lambda g: pd.Series({
    "사진 수": len(g),
    "권수 정확(%)": round((g["실제"] == g["검출"]).mean() * 100, 1),
    "권수 오차(평균)": round((g["실제"] - g["검출"]).abs().mean(), 2),
    "재현율(%)": round(g["맞음"].sum() / max(g["실제"].sum(), 1) * 100, 1),
    "정밀도(%)": round(g["맞음"].sum() / max(g["검출"].sum(), 1) * 100, 1),
})).reindex([o for o in order if o in set(ev["구간"])])
display(summary)
summary.to_csv(DRIVE / "eval_by_count.csv", encoding="utf-8-sig")
TIMES["권수 구간별 평가"] = (time.time() - t0) / 60
print(f"저장 → {DRIVE / 'eval_by_count.csv'}")

In [ ]:
# Cell 11 · 결과 한 번에 받기: 맥에 필요한 파일을 zip 하나로 묶어 Drive에 저장 + 브라우저로 내려받기
#   liber_det_results.zip
#     ├─ det_obb_best.pt          맥 weights/에 넣을 가중치
#     ├─ eval_by_count.csv        권수 구간별 평가
#     ├─ data_distribution.csv    사진 속 책 수 분포 · manifest.csv 출처별 목록
#     ├─ times.csv                단계별 걸린 시간
#     └─ run/                     학습 곡선(results.png·csv) · PR 곡선 · 혼동 행렬 · 검증 예측 그림 · 학습 설정(args.yaml)
#   (epoch*.pt·last.pt 같은 중간 가중치는 용량만 커서 뺌 — 필요하면 Drive runs/ 폴더에 있음)
from google.colab import files

BUNDLE = WORK / "liber_det_results"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
(BUNDLE / "run").mkdir(parents=True)
shutil.copy2(BEST_OUT, BUNDLE / "det_obb_best.pt")
for f in ("eval_by_count.csv",):
    if (DRIVE / f).exists():
        shutil.copy2(DRIVE / f, BUNDLE / f)
shutil.copy2(DS / "manifest.csv", BUNDLE / "manifest.csv")
dist.to_csv(BUNDLE / "data_distribution.csv", encoding="utf-8-sig")
times = pd.Series(TIMES, name="분").round(1)
times.to_csv(BUNDLE / "times.csv", encoding="utf-8-sig")
for f in RUN_DIR.iterdir():
    if f.is_file() and f.suffix in {".png", ".jpg", ".csv", ".yaml"}:
        shutil.copy2(f, BUNDLE / "run" / f.name)
zip_path = shutil.make_archive(str(WORK / "liber_det_results"), "zip", BUNDLE)
shutil.copy2(zip_path, DRIVE / "liber_det_results.zip")
print(f"묶음 저장 → {DRIVE / 'liber_det_results.zip'} ({Path(zip_path).stat().st_size / 1e6:.1f}MB)")
display(times.to_frame())
files.download(zip_path)   # 브라우저가 바로 내려받음 (막히면 Drive에서 받기)

## 맥북으로 가져가기

Cell 11이 내려받은 `liber_det_results.zip`(브라우저 다운로드 폴더, 또는 Drive `내 드라이브/liber/`)을 맥에서 한 번에 풀기:

```bash
cd ~/Downloads/shelf_ocr_test/liber_deprehensus
unzip -o ~/Downloads/liber_det_results.zip -d outputs/colab_det        # 평가표·학습 곡선·그림
cp outputs/colab_det/det_obb_best.pt weights/det_obb_best.pt           # 가중치 교체 (기존 1에폭 가중치를 덮어씀)
```

그다음 맥 노트북 커널 재시작 → Cell 0 → T → V → **1-4(가중치가 있으면 학습을 건너뜀)** → 1-5 → 1-6 …
   - 1-5는 가중치에 기록된 학습 입력 크기(960)를 자동으로 읽어 같은 크기로 검출한다

학습 곡선·검증 그림(`runs/det_obb/`)과 권수 구간별 평가(`eval_by_count.csv`)도 같은 폴더에 있다.